### Step 1 — Load Highline and LANDFIRE Vegetation Data

This notebook uses LANDFIRE to describe the vegetation that existed **before the focal wildfire**.

For the 2017 Highline fire, the LANDFIRE `v1_4_0` vegetation products represent conditions through approximately **2016**, making them an appropriate pre-fire vegetation snapshot.

Three products are loaded:

- **EVT — Existing Vegetation Type:** what kind of vegetation/ecosystem is present
- **EVC — Existing Vegetation Cover:** how much live vegetation/canopy cover is present
- **EVH — Existing Vegetation Height:** how tall the dominant vegetation is

Unlike Landsat, LANDFIRE is not a consistent annual time series. We therefore use the most appropriate **pre-fire LANDFIRE release** rather than constructing multi-year or Season 1 vegetation composites.

For the full 2013–2020 production pipeline, the appropriate historical LANDFIRE version will need to be selected for each fire year and cross-version comparability will be audited before scaling.

In [1]:
import ee

# Step 1: Initialize Earth Engine

ee.Initialize(project="ee-wildfire-recovery-vasant")

print("Earth Engine initialized")

# Reload the Highline focal fire independently

HIGHLINE_EVENT_ID = "ID4549011522720170729"

mtbs_boundaries = ee.FeatureCollection(
    "USFS/GTAC/MTBS/burned_area_boundaries/v1"
)

highline = ee.Feature(
    mtbs_boundaries
    .filter(ee.Filter.eq("Event_ID", HIGHLINE_EVENT_ID))
    .first()
)

print(
    "Fire:",
    highline.get("Incid_Name").getInfo(),
    "| Acres:",
    highline.get("BurnBndAc").getInfo(),
)


# Load LANDFIRE vegetation products

evt_collection = ee.ImageCollection(
    "LANDFIRE/Vegetation/EVT/v1_4_0"
)

evc_collection = ee.ImageCollection(
    "LANDFIRE/Vegetation/EVC/v1_4_0"
)

evh_collection = ee.ImageCollection(
    "LANDFIRE/Vegetation/EVH/v1_4_0"
)

print("\nEVT images:", evt_collection.size().getInfo())
print("EVC images:", evc_collection.size().getInfo())
print("EVH images:", evh_collection.size().getInfo())

print("\nEVT bands:", ee.Image(evt_collection.first()).bandNames().getInfo())
print("EVC bands:", ee.Image(evc_collection.first()).bandNames().getInfo())
print("EVH bands:", ee.Image(evh_collection.first()).bandNames().getInfo())

Earth Engine initialized
Fire: HIGHLINE | Acres: 93699

EVT images: 3
EVC images: 3
EVH images: 3

EVT bands: ['EVT']
EVC bands: ['EVC']
EVH bands: ['EVH']


#### Step 1 Output Interpretation

Each LANDFIRE vegetation product is stored as an Earth Engine `ImageCollection`.

The output:

- `EVT images: 3`
- `EVC images: 3`
- `EVH images: 3`

means that each collection contains **three geographic raster images**. These are not three years or three measurements; they are separate raster objects covering different geographic regions.

In Earth Engine, an `Image` is a **georeferenced raster grid**. Each 30 m pixel stores a value:

- `EVT` pixels store vegetation-type codes
- `EVC` pixels store vegetation-cover information
- `EVH` pixels store vegetation-height information

Each image contains one main band (`EVT`, `EVC`, or `EVH`).

The next step identifies which image in each collection actually covers the Highline fire.

### Step 2A

In [2]:
# Step 2A: Identify the LANDFIRE images covering Highline

evt_highline_collection = evt_collection.filterBounds(
    highline.geometry()
)

evc_highline_collection = evc_collection.filterBounds(
    highline.geometry()
)

evh_highline_collection = evh_collection.filterBounds(
    highline.geometry()
)

print(
    "EVT images intersecting Highline:",
    evt_highline_collection.size().getInfo()
)

print(
    "EVC images intersecting Highline:",
    evc_highline_collection.size().getInfo()
)

print(
    "EVH images intersecting Highline:",
    evh_highline_collection.size().getInfo()
)


# Inspect which geographic image(s) cover Highline
print(
    "\nEVT image IDs:",
    evt_highline_collection.aggregate_array(
        "system:index"
    ).getInfo()
)

print(
    "EVC image IDs:",
    evc_highline_collection.aggregate_array(
        "system:index"
    ).getInfo()
)

print(
    "EVH image IDs:",
    evh_highline_collection.aggregate_array(
        "system:index"
    ).getInfo()
)


# Select the image covering Highline
evt = ee.Image(
    evt_highline_collection.first()
).select("EVT")

evc = ee.Image(
    evc_highline_collection.first()
).select("EVC")

evh = ee.Image(
    evh_highline_collection.first()
).select("EVH")


# Inspect their native map grids
print("\nEVT projection:")
print(evt.projection().getInfo())

print("\nEVC projection:")
print(evc.projection().getInfo())

print("\nEVH projection:")
print(evh.projection().getInfo())

EVT images intersecting Highline: 1
EVC images intersecting Highline: 1
EVH images intersecting Highline: 1

EVT image IDs: ['CONUS']
EVC image IDs: ['CONUS']
EVH image IDs: ['CONUS']

EVT projection:
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.00026949458523585647, 0, -125.59606498707426, 0, -0.00026949458523585647, 49.612875163580235]}

EVC projection:
{'type': 'Projection', 'wkt': 'PROJCS["unnamed", \n  GEOGCS["NAD83", \n    DATUM["North_American_Datum_1983", \n      SPHEROID["GRS 1980", 6378137.0, 298.2572221010042, AUTHORITY["EPSG","7019"]], \n      TOWGS84[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], \n      AUTHORITY["EPSG","6269"]], \n    PRIMEM["Greenwich", 0.0], \n    UNIT["degree", 0.017453292519943295], \n    AXIS["Longitude", EAST], \n    AXIS["Latitude", NORTH], \n    AUTHORITY["EPSG","4269"]], \n  PROJECTION["Albers_Conic_Equal_Area"], \n  PARAMETER["central_meridian", -96.0], \n  PARAMETER["latitude_of_origin", 23.0], \n  PARAMETER["standard_parallel_1", 29.5], \n

### Step 2A — Output explanation

Filtering each LANDFIRE collection to the Highline fire leaves exactly one image:

- EVT → `CONUS`
- EVC → `CONUS`
- EVH → `CONUS`

`CONUS` is the raster covering the contiguous United States, so this is the geographic LANDFIRE image relevant to Highline.

The native grids are not represented identically in Earth Engine:

- **EVT** is stored on an `EPSG:4326` geographic grid with a pixel size roughly equivalent to 30 m.
- **EVC** and **EVH** are stored on 30 m Albers Equal Area grids.

This is not a problem because the final project will summarize each source geographically onto our explicit 90 m modeling patches rather than assuming that native pixel indices line up perfectly across products.

### Step 2B — Inspect LANDFIRE Values Inside Highline

The three LANDFIRE rasters store **coded classes**, not directly human-readable vegetation measurements.

#### EVT — Existing Vegetation Type
Highline contains **39 EVT classes**.

Each 4-digit value (for example `3056`, `3050`, or `3233`) identifies a particular ecological vegetation type. The codes themselves are not meaningful until they are matched to the LANDFIRE class names.

The large number of different EVT classes shows that the Highline perimeter contains substantial ecological diversity rather than one uniform forest type.

#### EVC — Existing Vegetation Cover
Highline contains **27 EVC classes**.

EVC values are cover classes rather than raw percentages. Different code ranges represent different dominant vegetation lifeforms such as trees, shrubs, and herbaceous vegetation, with additional classes describing increasing amounts of cover.

#### EVH — Existing Vegetation Height
Highline contains **12 EVH classes**.

EVH is also encoded into vegetation-height classes. A value such as `110` is therefore a class code and should not be interpreted as 110 meters.

The histogram frequencies are useful for identifying which classes are common, but the fractional values should not be treated as exact pixel counts. Later, class prevalence will be calculated using geographic area when exact proportions are needed.

In [3]:
# Step 2B: Inspect LANDFIRE values inside Highline

# EVT is categorical, so inspect which vegetation-type codes occur and how frequently they occur.
evt_hist = evt.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=highline.geometry(),
    scale=30,
    maxPixels=1e8,
)

# EVC and EVH may also use coded classes rather than behaving like simple continuous measurements, so inspect their value frequencies first as well.
evc_hist = evc.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=highline.geometry(),
    scale=30,
    maxPixels=1e8,
)

evh_hist = evh.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=highline.geometry(),
    scale=30,
    maxPixels=1e8,
)

evt_values = evt_hist.get("EVT").getInfo()
evc_values = evc_hist.get("EVC").getInfo()
evh_values = evh_hist.get("EVH").getInfo()

print("EVT unique classes:", len(evt_values))
print("EVT value frequencies:")
print(evt_values)

print("\nEVC unique values/classes:", len(evc_values))
print("EVC value frequencies:")
print(evc_values)

print("\nEVH unique values/classes:", len(evh_values))
print("EVH value frequencies:")
print(evh_values)

EVT unique classes: 39
EVT value frequencies:
{'3001': 254.84313725490196, '3006': 1052.517647058823, '3011': 20041.46274509804, '3045': 31711.886274509805, '3046': 4864.027450980392, '3047': 490, '3050': 46812.1764705882, '3053': 8118.596078431372, '3055': 39016.75294117649, '3056': 230878.8313725496, '3061': 40, '3079': 15, '3106': 1015.243137254902, '3123': 24, '3124': 735.5960784313725, '3125': 4, '3126': 2182.6196078431367, '3139': 7499.996078431376, '3140': 3466.7372549019606, '3145': 2390.9607843137255, '3159': 1797.3254901960784, '3160': 129.21176470588236, '3161': 8185.603921568624, '3164': 1390.0274509803921, '3166': 34351.68627450985, '3167': 10131.388235294122, '3168': 20, '3169': 3669.6901960784285, '3182': 379.6941176470588, '3220': 2103.1607843137253, '3227': 25106.125490196122, '3228': 97.7843137254902, '3233': 65864.77647058827, '3234': 2343.266666666667, '3235': 42469.223529411756, '3236': 6.372549019607844, '3251': 698.035294117647, '3252': 10, '3292': 159.9137254901

The next step decodes these stored class values into interpretable vegetation descriptions.

In [4]:
# Step 2B.2: Inspect common LANDFIRE classes and metadata

# Sort histogram dictionaries from most common to least common
def top_classes(histogram, n=15):
    return sorted(
        histogram.items(),
        key=lambda item: item[1],
        reverse=True
    )[:n]


print("Top 15 EVT codes in Highline:")
for code, frequency in top_classes(evt_values):
    print(code, "|", frequency)


print("\nTop 15 EVC codes in Highline:")
for code, frequency in top_classes(evc_values):
    print(code, "|", frequency)


print("\nTop 15 EVH codes in Highline:")
for code, frequency in top_classes(evh_values):
    print(code, "|", frequency)



# Inspect image properties to see what descriptive metadata LANDFIRE exposes directly through Earth Engine

print("\nEVT image properties:")
print(evt_highline_collection.first().propertyNames().getInfo())

print("\nEVC image properties:")
print(evc_highline_collection.first().propertyNames().getInfo())

print("\nEVH image properties:")
print(evh_highline_collection.first().propertyNames().getInfo())

Top 15 EVT codes in Highline:
3056 | 230878.8313725496
3233 | 65864.77647058827
3050 | 46812.1764705882
3235 | 42469.223529411756
3055 | 39016.75294117649
3166 | 34351.68627450985
3045 | 31711.886274509805
3227 | 25106.125490196122
3011 | 20041.46274509804
3167 | 10131.388235294122
3161 | 8185.603921568624
3053 | 8118.596078431372
3139 | 7499.996078431376
3046 | 4864.027450980392
3169 | 3669.6901960784285

Top 15 EVC codes in Highline:
103 | 141550.48627450978
104 | 109492.70196078435
102 | 80523.96862745105
105 | 36758.94117647058
101 | 25281.49019607845
106 | 8130.149019607845
122 | 3440.1019607843136
123 | 3319.1960784313724
114 | 1825.6156862745097
112 | 1776.4431372549022
113 | 1501.4823529411767
124 | 1235.0117647058823
121 | 1234.7529411764708
111 | 1208.0549019607843
115 | 1053.301960784314

Top 15 EVH codes in Highline:
110 | 346558.74509803933
109 | 25852.235294117658
108 | 16810.090196078432
111 | 13036.419607843132
101 | 10411.42745098039
104 | 4276.262745098039
106 | 2209.

In [5]:
# Step 2B.3: Decode EVT and EVC class codes using LANDFIRE metadata stored in Earth Engine


evt_image = ee.Image(evt_highline_collection.first())
evc_image = ee.Image(evc_highline_collection.first())
evh_image = ee.Image(evh_highline_collection.first())

# EVT code -> vegetation type name

evt_class_values = evt_image.get("EVT_class_values").getInfo()
evt_class_names = evt_image.get("EVT_class_names").getInfo()

evt_lookup = dict(
    zip(
        [str(value) for value in evt_class_values],
        evt_class_names
    )
)

print("Top 15 EVT classes in Highline:\n")

for code, frequency in top_classes(evt_values):
    print(
        code,
        "|",
        evt_lookup.get(code, "UNKNOWN CLASS"),
        "| frequency:",
        frequency
    )


# EVC code -> vegetation cover class name

evc_class_values = evc_image.get("EVC_class_values").getInfo()
evc_class_names = evc_image.get("EVC_class_names").getInfo()

evc_lookup = dict(
    zip(
        [str(value) for value in evc_class_values],
        evc_class_names
    )
)

print("\nTop 15 EVC classes in Highline:\n")

for code, frequency in top_classes(evc_values):
    print(
        code,
        "|",
        evc_lookup.get(code, "UNKNOWN CLASS"),
        "| frequency:",
        frequency
    )

# EVH does not expose equivalent class-name properties, so inspect its band metadata before decoding manually.

print("\nEVH band metadata:")
print(evh_image.get("system:bands").getInfo())

Top 15 EVT classes in Highline:

3056 | Rocky Mountain Subalpine Mesic-Wet Spruce-Fir Forest and Woodland | frequency: 230878.8313725496
3233 | Subalpine Douglas-fir Forest | frequency: 65864.77647058827
3050 | Rocky Mountain Lodgepole Pine Forest | frequency: 46812.1764705882
3235 | Xeric Montane Douglas-fir Forest | frequency: 42469.223529411756
3055 | Rocky Mountain Subalpine Dry-Mesic Spruce-Fir Forest and Woodland | frequency: 39016.75294117649
3166 | Middle Rocky Mountain Montane Douglas-fir Forest and Woodland | frequency: 34351.68627450985
3045 | Northern Rocky Mountain Dry-Mesic Montane Mixed Conifer Forest | frequency: 31711.886274509805
3227 | Dry-mesic Montane Douglas-fir Forest | frequency: 25106.125490196122
3011 | Rocky Mountain Aspen Forest and Woodland | frequency: 20041.46274509804
3167 | Rocky Mountain Poor-Site Lodgepole Pine Forest | frequency: 10131.388235294122
3161 | Northern Rocky Mountain Conifer Swamp | frequency: 8185.603921568624
3053 | Northern Rocky Mount

#### Decoded Vegetation Classes

LANDFIRE metadata successfully decoded the dominant EVT and EVC classes inside Highline.

**EVT:** Highline is dominated by forest and woodland ecological systems, particularly spruce-fir, Douglas-fir, lodgepole pine, mixed-conifer, and aspen systems. Smaller grassland, shrubland, sparse-vegetation, and parkland classes are also present. This supports using Highline as the forest-recovery POC while also showing why an explicit forest/woodland eligibility mask is necessary.

**EVC:** Most of the fire perimeter falls within tree-cover classes of approximately 20–60%, with smaller areas dominated by shrub and herbaceous cover. This indicates substantial pre-fire variation in canopy cover that may be useful as a model feature.

**EVH:** the inspected `system:bands` property confirms a native 30 m raster grid but does not contain descriptive height-class labels. The official Earth Engine catalog documents separate `EVH_classes` and `EVH_names` metadata properties, so these will be queried directly next.

The histogram frequencies are used here for relative prevalence only; exact class-area percentages will be calculated later using geographic area.


#### EVH Metadata Note

All three LANDFIRE products are rasters: geographic grids where each 30 m pixel stores a coded value.

The difference is in the **metadata attached to those rasters**:

- **EVT** includes built-in code → vegetation-type labels.
- **EVC** includes built-in code → cover-class labels.
- **EVH** contains the height-class raster itself, but the metadata inspected so far did not show an equally obvious code → height-label lookup.

So the EVH data is not missing; only its human-readable legend is less clearly exposed. The next check directly queries the documented EVH class/name properties before falling back to the official LANDFIRE legend.

In [6]:
# Step 2B.4: Retrieve documented EVH class metadata

evh_classes = evh_image.get("EVH_classes").getInfo()
evh_names = evh_image.get("EVH_names").getInfo()

print("EVH classes:")
print(evh_classes)

print("\nEVH names:")
print(evh_names)

EVH classes:
None

EVH names:
None


#### EVH Class-Label Lookup

To decode the EVH class codes, three possibilities were checked:

1. **Check the image metadata directly**  
   `propertyNames()` was used to see whether the EVH image contained built-in class-name/value metadata. Unlike EVT and EVC, no obvious EVH class lookup was listed.

2. **Query the documented EVH properties directly**  
   `EVH_classes` and `EVH_names` were requested explicitly in case they existed but were not shown by `propertyNames()`. Both returned `None`.

3. **Use the official LANDFIRE EVH class table**   (below)  

   Since the class legend is not embedded in the Earth Engine image, the EVH codes will be decoded using LANDFIRE's official external product documentation.

In [7]:
# Step 2B.5: Decode EVH codes using the official
# LANDFIRE Existing Vegetation Height legend

def decode_evh(code):
    code = int(code)

    # Special non-height classes observed in Highline
    if code == 11:
        return "Open Water"

    if code == 100:
        return "Sparse Vegetation Canopy"

    # LANDFIRE tree-height coding:
    # 101 = 1 m, 102 = 2 m, ..., 199 = >=99 m
    if 101 <= code <= 198:
        return f"Tree Height = {code - 100} m"

    if code == 199:
        return "Tree Height >= 99 m"

    return "Other / not decoded here"


print("EVH classes in Highline:\n")

for code, frequency in top_classes(evh_values, n=len(evh_values)):
    print(
        code,
        "|",
        decode_evh(code),
        "| frequency:",
        frequency
    )

EVH classes in Highline:

110 | Tree Height = 10 m | frequency: 346558.74509803933
109 | Tree Height = 9 m | frequency: 25852.235294117658
108 | Tree Height = 8 m | frequency: 16810.090196078432
111 | Tree Height = 11 m | frequency: 13036.419607843132
101 | Tree Height = 1 m | frequency: 10411.42745098039
104 | Tree Height = 4 m | frequency: 4276.262745098039
106 | Tree Height = 6 m | frequency: 2209.470588235294
100 | Sparse Vegetation Canopy | frequency: 909.392156862745
105 | Tree Height = 5 m | frequency: 896.1647058823528
102 | Tree Height = 2 m | frequency: 233.5372549019608
11 | Open Water | frequency: 110.64313725490196
107 | Tree Height = 7 m | frequency: 3


#### LANDFIRE Pixel Interpretation and Grid Alignment

For intuition, imagine one **~30 × 30 m LANDFIRE pixel**:

- **EVT:** *What vegetation type best describes this location?*  
  → `Lodgepole Pine Forest`

- **EVC:** *How much of the pixel is covered by the dominant vegetation?*  
  → `Tree cover 30–<40%`

- **EVH:** *What is the mapped average/representative height of the dominant vegetation in this pixel?*  
  → `~10 m`

EVH is also a ~30 m raster. A value such as `10 m` does **not** mean every tree is 10 m tall or that 10 m is the tallest tree present; it is LANDFIRE's modeled summary of dominant-vegetation height for that cell.

### Resolution vs. Grid

Although EVT, EVC, and EVH all have approximately **30 m spatial resolution**, their pixels are not necessarily positioned on exactly the same grid.

- **Resolution** describes roughly how large each pixel is.
- **Projection/grid** determines the coordinate system, pixel origin, orientation, and exact boundaries of those pixels.

For Highline:

- EVT is represented in a geographic latitude/longitude projection (`EPSG:4326`) with a pixel size approximately equivalent to 30 m.
- EVC and EVH use a 30 m Albers Equal Area grid.

Therefore, an EVT pixel and an EVC pixel may cover slightly different geographic footprints even though both are approximately 30 m.

This will be resolved later by creating our own fixed **90 × 90 m modeling grid**. Each source will be summarized independently according to its geographic overlap with the same 90 m patch rather than trying to match native source pixels directly.

Conceptually:

`native EVT/EVC/EVH grids → geographic overlap with common 90 m patch → patch-level vegetation features`

*This is a simplified mental model for understanding the products and how they will eventually be aligned.*

### Step 3 — LANDFIRE Visualization

In [8]:

# Step 3: Visualize EVT, EVC, and EVH across Highline

import geemap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

Map = geemap.Map()
Map.centerObject(highline, 10)


# 1. EVT — Existing Vegetation Type


# EVT is categorical. LANDFIRE's embedded Earth Engine palette
# contains many black entries, so for this POC we assign our own
# deterministic colors purely to distinguish classes visually.

highline_evt_codes = sorted(int(code) for code in evt_values.keys())

evt_display = evt.remap(
    highline_evt_codes,
    list(range(len(highline_evt_codes)))
)

cmap = plt.colormaps["gist_ncar"].resampled(len(highline_evt_codes))

highline_evt_palette = [
    mcolors.to_hex(cmap(i), keep_alpha=False).replace("#", "")
    for i in range(len(highline_evt_codes))
]

Map.addLayer(
    evt_display.clip(highline.geometry()),
    {
        "min": 0,
        "max": len(highline_evt_codes) - 1,
        "palette": highline_evt_palette,
    },
    "EVT — Vegetation Type"
)


# 2. EVC — Existing Vegetation Cover

evc_class_values = evc_image.get("EVC_class_values").getInfo()
evc_class_palette = evc_image.get("EVC_class_palette").getInfo()

evc_color_lookup = dict(
    zip(
        [int(v) for v in evc_class_values],
        evc_class_palette
    )
)

highline_evc_codes = sorted(int(code) for code in evc_values.keys())

highline_evc_palette = [
    evc_color_lookup[code]
    for code in highline_evc_codes
]

evc_display = evc.remap(
    highline_evc_codes,
    list(range(len(highline_evc_codes)))
)

Map.addLayer(
    evc_display.clip(highline.geometry()),
    {
        "min": 0,
        "max": len(highline_evc_codes) - 1,
        "palette": highline_evc_palette,
    },
    "EVC — Vegetation Cover"
)


# 3. EVH — Existing Vegetation Height

tree_height_m = (
    evh
    .updateMask(evh.gte(101).And(evh.lte(199)))
    .subtract(100)
    .rename("tree_height_m")
)

Map.addLayer(
    tree_height_m.clip(highline.geometry()),
    {
        "min": 1,
        "max": 11,
        "palette": [
            "fff7bc",
            "d9f0a3",
            "addd8e",
            "78c679",
            "41ab5d",
            "238443",
            "006837",
        ],
    },
    "EVH — Tree Height (m)"
)


# 4. Highline perimeter

highline_outline = ee.FeatureCollection([ee.Feature(highline)])

Map.addLayer(
    highline_outline.style(
        color="000000",
        fillColor="00000000",
        width=2
    ),
    {},
    "Highline Perimeter"
)

Map

Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

In [9]:
# Step 3B: Simplified EVT map — top 8 classes + Other

# Get the eight most common EVT classes in Highline.
top_evt = top_classes(evt_values, n=8)
top_evt_codes = [int(code) for code, _ in top_evt]

# Start every valid EVT pixel as "Other" (class 8).
evt_simple = (
    ee.Image.constant(8)
    .rename("EVT_group")
    .updateMask(evt.mask())
)

# Replace pixels belonging to the top classes with 0–7.
for i, code in enumerate(top_evt_codes):
    evt_simple = evt_simple.where(evt.eq(code), i)


# Distinct visualization colors.
evt_simple_palette = [
    "1f77b4",  # blue
    "ff7f0e",  # orange
    "2ca02c",  # green
    "d62728",  # red
    "9467bd",  # purple
    "8c564b",  # brown
    "e377c2",  # pink
    "17becf",  # cyan
    "bdbdbd",  # gray = Other
]


# Short labels for the interactive map legend.
evt_short_labels = [
    "3056 — Mesic-Wet Spruce-Fir",
    "3233 — Subalpine Douglas-fir",
    "3050 — Lodgepole Pine",
    "3235 — Xeric Montane Douglas-fir",
    "3055 — Dry-Mesic Spruce-Fir",
    "3166 — Montane Douglas-fir",
    "3045 — Mixed Conifer",
    "3227 — Dry-Mesic Douglas-fir",
    "Other EVT classes",
]

evt_legend = dict(
    zip(evt_short_labels, evt_simple_palette)
)


# Build map.
Map_evt_simple = geemap.Map()
Map_evt_simple.centerObject(highline, 10)

Map_evt_simple.addLayer(
    evt_simple.clip(highline.geometry()),
    {
        "min": 0,
        "max": 8,
        "palette": evt_simple_palette,
    },
    "EVT — Top 8 Types + Other"
)

Map_evt_simple.addLayer(
    highline_outline.style(
        color="000000",
        fillColor="00000000",
        width=2
    ),
    {},
    "Highline Perimeter"
)

# Add color legend directly to the map.
Map_evt_simple.add_legend(
    title="Major Pre-Fire Vegetation Types",
    legend_dict=evt_legend
)


# Keep the complete official names available below the map.
print("Full EVT legend:")
for i, code in enumerate(top_evt_codes):
    print(
        f"{i}: {code} — {evt_lookup[str(code)]}"
    )

print("8: Other EVT classes")

Map_evt_simple

Full EVT legend:
0: 3056 — Rocky Mountain Subalpine Mesic-Wet Spruce-Fir Forest and Woodland
1: 3233 — Subalpine Douglas-fir Forest
2: 3050 — Rocky Mountain Lodgepole Pine Forest
3: 3235 — Xeric Montane Douglas-fir Forest
4: 3055 — Rocky Mountain Subalpine Dry-Mesic Spruce-Fir Forest and Woodland
5: 3166 — Middle Rocky Mountain Montane Douglas-fir Forest and Woodland
6: 3045 — Northern Rocky Mountain Dry-Mesic Montane Mixed Conifer Forest
7: 3227 — Dry-mesic Montane Douglas-fir Forest
8: Other EVT classes


Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

EVT, EVC, and EVH were mapped across the Highline fire perimeter to visually inspect the pre-fire vegetation landscape.

- **EVT** showed substantial spatial variation in vegetation type. The Earth Engine metadata palette contained many black color entries, which made the raw official palette unusable for interpretation. We therefore kept the official EVT class codes/names but assigned our own categorical colors for visualization.
- **EVC** showed mostly moderate tree-cover classes with smaller contrasting patches, consistent with the earlier class-frequency results.
- **EVH** was relatively uniform because Highline is strongly dominated by the ~10 m tree-height class, with smaller areas of shorter vegetation.

For easier EVT interpretation, a second map grouped the **8 most common vegetation types plus an `Other` category** and added a color legend.

The visualization colors are presentation choices only; the underlying LANDFIRE values and scientific class meanings remain unchanged.

### Step 4 — Define Pre-Fire Forest/Woodland Eligibility

The goal of this step is to create a **binary pre-fire forest/woodland mask** so that the final modeling population contains only locations that were forested or wooded before the focal wildfire.

Highline is used as the **POC test case** to develop and validate the classification logic. The rule should be conceptually generalizable, but the complete LANDFIRE class lookup and cross-version harmonization will be handled when the pipeline is scaled in Phase 3.

Broadly, this step will:

1. Identify which EVT vegetation types qualify as forest/woodland.
2. Build the eligibility mask.
3. Quantify and visually validate the resulting forested area.
4. Document the rule and remaining production-level audits.

In [10]:
# Step 4A.1: Inspect all EVT classes present in Highline

highline_evt_classes = sorted(
    [(int(code), freq) for code, freq in evt_values.items()],
    key=lambda x: x[1],
    reverse=True
)

print(f"EVT classes present in Highline: {len(highline_evt_classes)}\n")

for rank, (code, freq) in enumerate(highline_evt_classes, start=1):
    label = evt_lookup[str(code)]

    print(
        f"{rank:>2}. "
        f"{code} — {label}"
    )

EVT classes present in Highline: 39

 1. 3056 — Rocky Mountain Subalpine Mesic-Wet Spruce-Fir Forest and Woodland
 2. 3233 — Subalpine Douglas-fir Forest
 3. 3050 — Rocky Mountain Lodgepole Pine Forest
 4. 3235 — Xeric Montane Douglas-fir Forest
 5. 3055 — Rocky Mountain Subalpine Dry-Mesic Spruce-Fir Forest and Woodland
 6. 3166 — Middle Rocky Mountain Montane Douglas-fir Forest and Woodland
 7. 3045 — Northern Rocky Mountain Dry-Mesic Montane Mixed Conifer Forest
 8. 3227 — Dry-mesic Montane Douglas-fir Forest
 9. 3011 — Rocky Mountain Aspen Forest and Woodland
10. 3167 — Rocky Mountain Poor-Site Lodgepole Pine Forest
11. 3161 — Northern Rocky Mountain Conifer Swamp
12. 3053 — Northern Rocky Mountain Ponderosa Pine Woodland and Savanna
13. 3139 — Northern Rocky Mountain Lower Montane-Foothill-Valley Grassland
14. 3046 — Northern Rocky Mountain Subalpine Woodland and Parkland
15. 3169 — Northern Rocky Mountain Subalpine Deciduous Shrubland
16. 3140 — Northern Rocky Mountain Subalpine-

In [11]:
# Step 4A.2: Candidate forest/woodland classification

# Clearly forest / woodland / tree-dominated systems.
forest_evt_codes = [
    3056,  # Spruce-Fir Forest and Woodland
    3233,  # Subalpine Douglas-fir Forest
    3050,  # Lodgepole Pine Forest
    3235,  # Xeric Montane Douglas-fir Forest
    3055,  # Spruce-Fir Forest and Woodland
    3166,  # Montane Douglas-fir Forest and Woodland
    3045,  # Mixed Conifer Forest
    3227,  # Douglas-fir Forest
    3011,  # Aspen Forest and Woodland
    3167,  # Lodgepole Pine Forest
    3053,  # Ponderosa Pine Woodland and Savanna
    3046,  # Subalpine Woodland and Parkland
    3234,  # Mesic Montane Douglas-fir Forest
    3159,  # Montane Riparian Forest and Woodland
    3047,  # Mesic Montane Mixed Conifer Forest
    3160,  # Subalpine/Upper Montane Riparian Forest and Woodland
    3228,  # Western Larch Forest
    3061,  # Aspen-Mixed Conifer Forest and Woodland
    3236,  # Subalpine Western Larch Forest
]

# Potentially forested, but ecologically distinct enough to review it before locking the mask.
review_evt_codes = [
    3161,  # Northern Rocky Mountain Conifer Swamp
]

# Everything else observed in Highline is non-forest/non-woodland.
exclude_evt_codes = [
    code
    for code, _ in highline_evt_classes
    if code not in forest_evt_codes + review_evt_codes
]

print(f"KEEP:    {len(forest_evt_codes)} classes")
print(f"REVIEW:  {len(review_evt_codes)} classes")
print(f"EXCLUDE: {len(exclude_evt_codes)} classes")

print("\nREVIEW:")
for code in review_evt_codes:
    print(f"{code} — {evt_lookup[str(code)]}")

print("\nEXCLUDE:")
for code in exclude_evt_codes:
    print(f"{code} — {evt_lookup[str(code)]}")

KEEP:    19 classes
REVIEW:  1 classes
EXCLUDE: 19 classes

REVIEW:
3161 — Northern Rocky Mountain Conifer Swamp

EXCLUDE:
3139 — Northern Rocky Mountain Lower Montane-Foothill-Valley Grassland
3169 — Northern Rocky Mountain Subalpine Deciduous Shrubland
3140 — Northern Rocky Mountain Subalpine-Upper Montane Grassland
3145 — Rocky Mountain Subalpine-Montane Mesic Meadow
3126 — Inter-Mountain Basins Montane Sagebrush Steppe
3220 — Artemisia tridentata ssp. vaseyana Shrubland Alliance
3164 — Rocky Mountain Wetland-Herbaceous
3006 — Rocky Mountain Alpine/Montane Sparsely Vegetated Systems
3106 — Northern Rocky Mountain Montane-Foothill Deciduous Shrubland
3124 — Columbia Plateau Low Sagebrush Steppe
3251 — Rocky Mountain Montane Riparian Shrubland
3182 — Introduced Upland Vegetation-Perennial Grassland and Forbland
3001 — Inter-Mountain Basins Sparsely Vegetated Systems
3292 — Open Water
3123 — Columbia Plateau Steppe and Grassland
3168 — Northern Rocky Mountain Avalanche Chute Shrubland


#### Conifer Swamp Decision

`3161 — Northern Rocky Mountain Conifer Swamp` is **kept** because it is still a tree-dominated forest/woodland system, even though it occurs in wetland conditions.

The wet environment may affect recovery behavior, but that ecological difference is useful variation for the model rather than a reason to exclude the class.

In [12]:
# Step 4A.3: Finalize Highline forest/woodland EVT classes

forest_evt_codes.append(3161)
review_evt_codes = []

print(f"Final KEEP: {len(forest_evt_codes)} classes")
print(f"Final EXCLUDE: {len(exclude_evt_codes)} classes")
print(f"Remaining REVIEW: {len(review_evt_codes)} classes")

Final KEEP: 20 classes
Final EXCLUDE: 19 classes
Remaining REVIEW: 0 classes


In [13]:
# Step 4B: Build binary pre-fire forest/woodland mask

forest_mask = (
    evt
    .remap(
        forest_evt_codes,
        [1] * len(forest_evt_codes),
        0
    )
    .rename("forest_woodland")
    .clip(highline.geometry())
)

print("Forest/woodland mask created.")
print("Band:", forest_mask.bandNames().getInfo())

Forest/woodland mask created.
Band: ['forest_woodland']


#### Step 4C — Quantify and Validate the Forest/Woodland Mask

Using `ee.Image.pixelArea()`, the EVT-derived eligibility mask was converted into actual geographic area within the Highline fire perimeter.

- Forest/woodland: **362.05 km²**
- Non-forest: **17.12 km²**
- Valid LANDFIRE area: **379.18 km²**
- Forest/woodland fraction: **95.48%**

`pixelArea()` was used instead of treating histogram frequencies as literal pixel counts, allowing forest and non-forest coverage to be measured directly in square meters.

The binary mask was also mapped for visual validation. Highline is overwhelmingly forest/woodland, with only about **4.5%** of the LANDFIRE-covered fire area excluded as grassland, shrubland, meadow, sparse vegetation, water, or other non-forest systems.

In [14]:
# Step 4C.1: Quantify forest/woodland eligibility

pixel_area = ee.Image.pixelArea()

forest_area_m2 = (
    pixel_area
    .updateMask(forest_mask.eq(1))
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=highline.geometry(),
        scale=30,
        maxPixels=1e8
    )
    .get("area")
    .getInfo()
)

nonforest_area_m2 = (
    pixel_area
    .updateMask(forest_mask.eq(0))
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=highline.geometry(),
        scale=30,
        maxPixels=1e8
    )
    .get("area")
    .getInfo()
)

valid_area_m2 = forest_area_m2 + nonforest_area_m2
forest_pct = 100 * forest_area_m2 / valid_area_m2

print(f"Forest/woodland area:     {forest_area_m2 / 1e6:,.2f} km²")
print(f"Non-forest area:          {nonforest_area_m2 / 1e6:,.2f} km²")
print(f"Valid LANDFIRE area:      {valid_area_m2 / 1e6:,.2f} km²")
print(f"Forest/woodland fraction: {forest_pct:.2f}%")

Forest/woodland area:     362.05 km²
Non-forest area:          17.12 km²
Valid LANDFIRE area:      379.18 km²
Forest/woodland fraction: 95.48%


In [15]:
# Step 4C.2: Visualize forest/woodland eligibility mask

Map_forest = geemap.Map()
Map_forest.centerObject(highline, 10)

Map_forest.addLayer(
    forest_mask,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "d9d9d9",  # 0 = non-forest
            "228B22",  # 1 = forest/woodland
        ],
    },
    "Pre-Fire Forest/Woodland Mask"
)

Map_forest.addLayer(
    highline_outline.style(
        color="000000",
        fillColor="00000000",
        width=2
    ),
    {},
    "Highline Perimeter"
)

Map_forest.add_legend(
    title="Pre-Fire Eligibility",
    legend_dict={
        "Forest / Woodland": "228B22",
        "Non-Forest": "d9d9d9",
    }
)

Map_forest

Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

#### Note: Forest/Woodland Eligibility Rule

For the Highline POC, pre-fire forest/woodland eligibility is derived from **LANDFIRE EVT**:

- `1` = EVT class represents a tree-dominated forest/woodland system.
- `0` = shrubland, grassland, meadow, herbaceous wetland, sparse vegetation, open water, or other non-forest system.

The Highline POC contains **20 eligible forest/woodland EVT classes** and 19 excluded classes. This retains **95.48%** of the LANDFIRE-covered fire area.

The classification principle is intended to generalize, but the current code list is **Highline-specific**. During Phase 3, the rule must be expanded and audited against the full relevant LANDFIRE EVT taxonomy and historical LANDFIRE versions used across 2013–2020 fires.

The original EVT data remain unchanged; the binary mask is a derived eligibility layer used to define the modeling population.

### Step 5 — Define LANDFIRE Features for Future 90 m Patches

The common **90 × 90 m modeling grid will be created later during the Phase 2 integration step**.

The goal here is therefore to define what LANDFIRE should eventually contribute to each 90 m modeling row and how each source should be summarized once that common grid exists.

This step will:

1. Define the proposed LANDFIRE feature set.
2. Validate that EVT, EVC, EVH, and the forest mask are represented in sensible ways.
3. Document how these layers should be aggregated onto the future 90 m grid, while leaving nationwide class harmonization and final threshold choices for Phase 3.

#### Step 5A — Proposed LANDFIRE Feature Contract

For the initial modeling pipeline, LANDFIRE will contribute five interpretable pre-fire vegetation features to each future 90 × 90 m patch:

- **`forest_fraction`** — fraction of the entire patch classified as pre-fire forest/woodland.
- **`dominant_evt`** — forest/woodland EVT class covering the largest forested area in the patch.
- **`dominant_evt_fraction`** — fraction of the forested area represented by that dominant EVT class.
- **`tree_cover_mean`** — area-weighted mean pre-fire tree-canopy cover over the forested portion of the patch.
- **`tree_height_mean`** — area-weighted mean mapped tree height over the forested portion of the patch.

These features were chosen to represent distinct ecological concepts—forest presence, vegetation type/composition, canopy density, and vegetation structure—without prematurely creating a large number of potentially redundant features.

This is a **parsimonious first-pass feature set**, not a claim that five features are optimal. During Phase 3/EDA, additional features such as cover/height variability, vegetation diversity, or broader ecosystem-composition proportions can be tested and retained only if they add useful information or predictive value.

#### Converting EVC to Numeric Tree Cover

LANDFIRE EVC stores canopy cover as coded percentage ranges rather than continuous percentages. Tree-cover classes were therefore converted to their class midpoints (for example, `30–<40% → 35%`) to create a numeric `tree_cover_pct` raster.

This allows meaningful area-weighted averaging when EVC is eventually summarized onto 90 m patches. The raw EVC class codes themselves are not averaged because codes such as `103` and `104` are identifiers, not canopy-cover measurements.

Only **Tree Cover** classes are converted; shrub and herb cover classes are not assigned artificial tree-cover values.

In [16]:
# Step 5A.1: Convert EVC tree-cover classes to numeric midpoint percentages

import re


def tree_cover_midpoint(label):
    """
    Convert a LANDFIRE Tree Cover class label into an
    approximate numeric canopy-cover percentage.

    Examples:
        >=30 <40%  -> 35%
        >=40 <50%  -> 45%
    """

    if not label.startswith("Tree Cover"):
        return None

    numbers = [
        float(x)
        for x in re.findall(r"\d+(?:\.\d+)?", label)
    ]

    # Standard bounded class, e.g. >=30 <40%
    if len(numbers) >= 2:
        return (numbers[0] + numbers[1]) / 2

    # Handle an open-ended upper class if one exists, using 100% as the natural upper bound for canopy cover.
    if len(numbers) == 1 and ">=" in label:
        return (numbers[0] + 100) / 2

    # Handle a possible lower class such as <10%.
    if len(numbers) == 1 and "<" in label:
        return numbers[0] / 2

    return None


# Build code -> midpoint lookup using LANDFIRE metadata.
tree_cover_lookup = {}

for code_str, label in evc_lookup.items():

    midpoint = tree_cover_midpoint(label)

    if midpoint is not None:
        tree_cover_lookup[int(code_str)] = midpoint


# Show only tree-cover classes actually observed in Highline.
print("Highline EVC tree-cover conversion:\n")

for code in sorted(highline_evc_codes):

    if code in tree_cover_lookup:
        print(
            f"{code} — {evc_lookup[str(code)]}"
            f"  →  {tree_cover_lookup[code]:.1f}%"
        )


# Create a numeric tree-cover raster.
# Non-tree EVC classes are left masked.
tree_cover_pct = (
    evc
    .remap(
        list(tree_cover_lookup.keys()),
        list(tree_cover_lookup.values())
    )
    .rename("tree_cover_pct")
    .updateMask(forest_mask.eq(1))
)

print("\nNumeric tree-cover image created.")
print("Band:", tree_cover_pct.bandNames().getInfo())

Highline EVC tree-cover conversion:

101 — Tree Cover >= 10 and < 20%  →  15.0%
102 — Tree Cover >= 20 and < 30%  →  25.0%
103 — Tree Cover >= 30 and < 40%  →  35.0%
104 — Tree Cover >= 40 and < 50%  →  45.0%
105 — Tree Cover >= 50 and < 60%  →  55.0%
106 — Tree Cover >= 60 and < 70%  →  65.0%
107 — Tree Cover >= 70 and < 80%  →  75.0%
108 — Tree Cover >= 80 and < 90%  →  85.0%
109 — Tree Cover >= 90 and <= 100%  →  95.0%

Numeric tree-cover image created.
Band: ['tree_cover_pct']


#### Step 5B — Validate LANDFIRE Feature Consistency

Before using these variables in future 90 × 90 m patches, verify that the LANDFIRE layers agree sensibly with each other.

In particular, check whether areas classified as forest/woodland by EVT also contain valid **tree-cover information from EVC** and **tree-height information from EVH**. Any systematic disagreement would need to be handled before aggregation into modeling features.

In [17]:
# Step 5B.1: Check EVC and EVH coverage within eligible forest

# Area classified as forest/woodland by EVT.
forest_area = pixel_area.updateMask(forest_mask.eq(1))


# Forest area with a valid numeric tree-cover value.
forest_with_cover_area_m2 = (
    pixel_area
    .updateMask(
        forest_mask.eq(1)
        .And(tree_cover_pct.mask().gt(0))
    )
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=highline.geometry(),
        scale=30,
        maxPixels=1e8
    )
    .get("area")
    .getInfo()
)


# Forest area with a valid numeric tree-height value.
forest_with_height_area_m2 = (
    pixel_area
    .updateMask(
        forest_mask.eq(1)
        .And(tree_height_m.mask().gt(0))
    )
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=highline.geometry(),
        scale=30,
        maxPixels=1e8
    )
    .get("area")
    .getInfo()
)


cover_pct_of_forest = (
    100 * forest_with_cover_area_m2 / forest_area_m2
)

height_pct_of_forest = (
    100 * forest_with_height_area_m2 / forest_area_m2
)


print(
    f"Eligible forest area:             "
    f"{forest_area_m2 / 1e6:,.2f} km²"
)

print(
    f"Forest with valid tree cover:     "
    f"{forest_with_cover_area_m2 / 1e6:,.2f} km² "
    f"({cover_pct_of_forest:.2f}%)"
)

print(
    f"Forest with valid tree height:    "
    f"{forest_with_height_area_m2 / 1e6:,.2f} km² "
    f"({height_pct_of_forest:.2f}%)"
)

Eligible forest area:             362.05 km²
Forest with valid tree cover:     358.87 km² (99.12%)
Forest with valid tree height:    361.86 km² (99.95%)


##### Step 5B.2 — Inspect Small Cross-Layer Disagreements

EVT, EVC, and EVH are separate LANDFIRE products, so small spatial disagreements are expected.

This check identifies locations where **EVT classifies the area as forest/woodland**, but the corresponding EVC or EVH layer does not provide a usable tree-cover or tree-height value. The goal is to understand the source of these small mismatches rather than assume the layers agree perfectly.

In [18]:
# Step 5B.2: Inspect small EVT–EVC/EVH disagreements

# Forest according to EVT, but no numeric tree-cover value.
cover_mismatch = (
    forest_mask.eq(1)
    .And(tree_cover_pct.mask().eq(0))
)

cover_mismatch_hist = (
    evc.updateMask(cover_mismatch)
    .reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=highline.geometry(),
        scale=30,
        maxPixels=1e8
    )
    .get("EVC")
    .getInfo()
)

print("EVC classes where EVT = forest but tree cover is unavailable:\n")

if cover_mismatch_hist:
    for code, freq in sorted(
        cover_mismatch_hist.items(),
        key=lambda x: x[1],
        reverse=True
    ):
        print(
            f"{code} — {evc_lookup.get(str(code), 'Unknown class')}"
            f" | frequency: {freq}"
        )
else:
    print("None")


print("\nEVH codes where EVT = forest but tree height is unavailable:\n")

height_mismatch = (
    forest_mask.eq(1)
    .And(tree_height_m.mask().eq(0))
)

height_mismatch_hist = (
    evh.updateMask(height_mismatch)
    .reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=highline.geometry(),
        scale=30,
        maxPixels=1e8
    )
    .get("EVH")
    .getInfo()
)

if height_mismatch_hist:
    for code, freq in sorted(
        height_mismatch_hist.items(),
        key=lambda x: x[1],
        reverse=True
    ):
        print(f"{code} | frequency: {freq}")
else:
    print("None")

EVC classes where EVT = forest but tree cover is unavailable:

122 — Herb Cover >= 20 and < 30% | frequency: 616.9490196078431
123 — Herb Cover >= 30 and < 40% | frequency: 552.8862745098039
112 — Shrub Cover >= 20 and < 30% | frequency: 402.83137254901953
111 — Shrub Cover >= 10 and < 20% | frequency: 296.5529411764706
114 — Shrub Cover >= 40 and < 50% | frequency: 294.3607843137255
121 — Herb Cover >= 10 and < 20% | frequency: 277.8627450980392
113 — Shrub Cover >= 30 and < 40% | frequency: 268.1372549019608
125 — Herb Cover >= 50 and < 60% | frequency: 230.94901960784313
124 — Herb Cover >= 40 and < 50% | frequency: 226.1333333333333
100 — Sparse Vegetation Canopy | frequency: 175.88235294117644
126 — Herb Cover >= 60 and < 70% | frequency: 153
115 — Shrub Cover >= 50 and < 60% | frequency: 150
11 — Open Water | frequency: 40.14509803921569
128 — Herb Cover >= 80 and < 90% | frequency: 5
117 — Shrub Cover >= 70 and < 80% | frequency: 4
118 — Shrub Cover >= 80 and < 90% | frequency: 

##### Step 5B.2 — Interpreting Cross-Layer Disagreements

Small EVT–EVC/EVH disagreements are expected because the three products describe different aspects of vegetation.

- **EVT** defines the ecological vegetation type and is therefore used to determine forest/woodland eligibility.
- **EVC** describes the dominant vegetation cover and provides tree-cover information where a valid `Tree Cover` class exists.
- **EVH** provides tree-height information where a valid `Tree Height` class exists.

Therefore, an EVT forest pixel with shrub/herb EVC is not automatically contradictory: it can belong to a forest/woodland ecosystem while shrubs or herbs dominate that particular location.

For Highline, agreement is very high:
- **99.12%** of eligible forest has valid tree-cover information.
- **99.95%** has valid tree-height information.

Small mismatched areas are **not automatically removed or converted to zero**. Tree-cover/height summaries will use the valid portions of each patch, while the amount of valid coverage can be tracked as a QA measure.

During Phase 3, the full dataset will be used to determine whether a minimum valid-coverage threshold is needed. Patches with substantial disagreement may then be flagged or excluded rather than choosing an arbitrary threshold during the POC.

#### Step 5C.1 — LANDFIRE Integration Handoff

When the common **90 × 90 m modeling grid** is created during Phase 2 integration, LANDFIRE will be summarized onto each patch as follows:

- **`forest_fraction`** = forest/woodland area ÷ total patch area.
- **`dominant_evt`** = EVT class covering the largest portion of the patch's forested area.
- **`dominant_evt_fraction`** = dominant EVT area ÷ total forested area.
- **`tree_cover_mean`** = area-weighted mean numeric EVC tree-cover percentage over valid forested area.
- **`tree_height_mean`** = area-weighted mean EVH tree height over valid forested area.

EVT remains categorical and is never numerically averaged. EVC and EVH are summarized numerically only after being converted into meaningful physical values.

Small EVC/EVH gaps within EVT-defined forest are allowed; valid coverage will be tracked as a QA measure rather than automatically treating missing structural information as zero.

#### Step 5C.2 — Phase 3 Decisions Still Open

The LANDFIRE POC establishes the feature logic, but several production decisions should be made using the full multi-fire dataset rather than fixed arbitrarily now:

- Minimum **`forest_fraction`** required for a 90 m patch to enter the modeling population.
- Minimum valid EVC/EVH coverage required for reliable `tree_cover_mean` and `tree_height_mean`.
- Whether additional heterogeneity features such as cover/height variability or vegetation diversity improve the model.
- Generalization of the forest/woodland EVT classification beyond Highline to the full relevant Western U.S. taxonomy.
- Selection and harmonization of historical LANDFIRE versions across the 2013–2020 fire years.
- Final handling of rare or ambiguous cross-layer disagreements.

These choices will be evaluated during dataset construction and EDA rather than being tuned specifically to the Highline POC.

In [19]:
# Step 6A: Save curated LANDFIRE POC figures

from pathlib import Path
import urllib.request


# Find repository root.
cwd = Path.cwd()
repo_root = cwd if (cwd / "pyproject.toml").exists() else cwd.parent

figure_dir = repo_root / "outputs" / "figures" / "landfire"
figure_dir.mkdir(parents=True, exist_ok=True)


# Highline perimeter overlay.
outline = (
    ee.Image()
    .byte()
    .paint(highline_outline, 1, 2)
    .selfMask()
    .visualize(palette=["000000"])
)


def save_ee_png(image, vis_params, filename):
    """Visualize an EE image, add the Highline outline, and save as PNG."""

    rendered = image.clip(highline.geometry()).visualize(**vis_params)
    rendered = rendered.blend(outline)

    url = rendered.getThumbURL({
        "region": highline.geometry(),
        "dimensions": 1600,
        "format": "png",
    })

    output_path = figure_dir / filename
    urllib.request.urlretrieve(url, output_path)

    print(f"Saved: {output_path.relative_to(repo_root)}")


# 1. Simplified EVT: top 8 vegetation types + Other
save_ee_png(
    evt_simple,
    {
        "min": 0,
        "max": 8,
        "palette": evt_simple_palette,
    },
    "highline_evt_top8.png",
)


# 2. EVC
save_ee_png(
    evc_display,
    {
        "min": 0,
        "max": len(highline_evc_codes) - 1,
        "palette": highline_evc_palette,
    },
    "highline_evc.png",
)


# 3. EVH tree height
save_ee_png(
    tree_height_m,
    {
        "min": 1,
        "max": 11,
        "palette": [
            "fff7bc",
            "d9f0a3",
            "addd8e",
            "78c679",
            "41ab5d",
            "238443",
            "006837",
        ],
    },
    "highline_evh.png",
)


# 4. Forest/woodland eligibility mask
save_ee_png(
    forest_mask,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "d9d9d9",
            "228B22",
        ],
    },
    "highline_forest_mask.png",
)

Saved: outputs\figures\landfire\highline_evt_top8.png
Saved: outputs\figures\landfire\highline_evc.png
Saved: outputs\figures\landfire\highline_evh.png
Saved: outputs\figures\landfire\highline_forest_mask.png
